In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import Window, functions as F


In [3]:
S3_BUCKET = "datalake-teste2"
S3_BRONZE = f"s3a://{S3_BUCKET}/bronze"
S3_SILVER = f"s3a://{S3_BUCKET}/silver"
S3_GOLD = f"s3a://{S3_BUCKET}/gold"

SPARK_MASTER = "local[*]"
SPARK_APP_NAME = "medallion-pipeline"

print(f"Bronze:  {S3_BRONZE}")
print(f"Silver:  {S3_SILVER}")
print(f"Gold:    {S3_GOLD}")

Bronze:  s3a://datalake-teste2/bronze
Silver:  s3a://datalake-teste2/silver
Gold:    s3a://datalake-teste2/gold


In [4]:
spark = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .master(SPARK_MASTER) \
    .config("spark.sql.session.timeZone", "UTC") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:4566") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.2,com.amazonaws:aws-java-sdk-bundle:1.12.261") \
    .getOrCreate()

26/09/03 03:08:42 WARN Utils: Your hostname, hugo resolves to a loopback address: 127.0.1.1; using 10.159.141.203 instead (on interface wlp2s0)
26/09/03 03:08:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/hugo/Desktop/Project/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hugo/.ivy2/cache
The jars for the packages stored in: /home/hugo/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-750f314e-b13e-490f-8f5d-3e39a61cd9a1;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.261 in central
:: resolution report :: resolve 387ms :: artifacts dl 16ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.261 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 by [com.amazonaws#aws-java-sdk-bundle;1.12.261] in [default]
	------------------------------------------------------------------

In [5]:
df_purchase_historico = spark.read.parquet(f"{S3_GOLD}/purchase_historico")

df_purchase_historico.createOrReplaceTempView("purchase_historico")

26/09/03 03:08:47 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [6]:
df_purchase_historico.show()

+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+--------------------+------------------+---------------+----------+----------------+
|transaction_datetime|purchase_id|buyer_id|prod_item_id|order_date|release_date|producer_id|product_id|item_quantity|purchase_value|   subsidiary|       fontes_no_dia|dt_inicio_validade|dt_fim_validade|is_current|transaction_date|
+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+--------------------+------------------+---------------+----------+----------------+
| 2023-03-12 07:00:00|         69|  160001|          18|2023-02-26|  2023-02-28|      96967|    373737|            2|       2000.00|internacional|[purchase_extra_i...|        2023-03-12|           NULL|      true|      2023-03-12|
| 2023-09-01 06:00:00|         70|  220134|          21|2023-03-02|  2023-03

O que está sendo feito?

- GMV DIARIO

In [7]:
GMV_DIARIO = """
    SELECT release_date                           AS dia,
           COALESCE(subsidiary, 'não informada')  AS subsidiaria,
           SUM(purchase_value)                    AS gmv
    FROM purchase_historico
    WHERE is_current
      AND release_date IS NOT NULL
    GROUP BY 1, 2
    ORDER BY 1, 2
"""

print("=" * 60)
print("💰 GMV diário por subsidiária — dados correntes de hoje")
print("=" * 60)

spark.sql(GMV_DIARIO).show(100, truncate=False)

💰 GMV diário por subsidiária — dados correntes de hoje


+----------+-------------+-------+
|dia       |subsidiaria  |gmv    |
+----------+-------------+-------+
|2023-02-28|internacional|2000.00|
|2023-03-01|nacional     |55.00  |
|2023-03-02|internacional|330.00 |
|2023-03-19|internacional|750.00 |
|2023-04-18|não informada|640.00 |
|2023-05-22|internacional|480.00 |
|2023-06-08|nacional     |900.00 |
|2023-06-25|internacional|150.00 |
+----------+-------------+-------+



In [8]:
# Mesma consulta, mas como o GMV era conhecido numa data passada. Só o filtro
# de vigência muda. Como nenhuma partição passada é reescrita, o resultado é
# sempre o mesmo, não importa quando a consulta seja executada.
GMV_EM_UMA_DATA = """
                SELECT release_date                           AS dia,
                       COALESCE(subsidiary, 'não informada')  AS subsidiaria,
                       SUM(purchase_value)                    AS gmv
                FROM purchase_historico
                WHERE date'{data}' BETWEEN dt_inicio_validade
                                       AND COALESCE(dt_fim_validade, date'9999-12-31')
                  AND release_date IS NOT NULL
                GROUP BY 1, 2
                ORDER BY 1, 2
                """

print("=" * 60)
print("🕓 O mesmo GMV, como era conhecido em 31/03/2023")
print("=" * 60)
spark.sql(GMV_EM_UMA_DATA.format(data="2023-03-31")).show(100, truncate=False)

🕓 O mesmo GMV, como era conhecido em 31/03/2023


+----------+-------------+-------+
|dia       |subsidiaria  |gmv    |
+----------+-------------+-------+
|2023-01-20|nacional     |50.00  |
|2023-02-28|internacional|2000.00|
|2023-03-02|nacional     |300.00 |
|2023-03-19|internacional|750.00 |
+----------+-------------+-------+

